# 04

Notebook to get covariates for each donor for tensorQTL

In [1]:
import pandas as pd
import os

## Donor info covs

In [ ]:
# add metadata
metadata_path = '/ASA/analysis/donor_statistics/crn_metadata/upload/SUBJECT.csv'
metadata = pd.read_csv(metadata_path)
metadata_sample_path = '/ASA/analysis/donor_statistics/crn_metadata/upload/SAMPLE.csv'
metadata_sample = pd.read_csv(metadata_sample_path)

In [ ]:
metadata = metadata.merge(metadata_sample, how = 'left', left_on = 'subject_id', right_on = 'sample_id')
metadata = metadata.rename(columns={'subject_id_x': 'subject_id'})
metadata

## Method info covs

In [ ]:
model = pd.read_csv('/ASA/analysis/WGS_mod/modkit_cram/metadata/metadata_model.csv')
model

,sample_name,kit,method,model,mod_model,basecaller
0,ASA_001B,SQK-LSK114,Native,dna_r10.4.1_e8.2_400bps_sup@v5.0.0,5mC_5hmC,Dorado
1,ASA_002B,SQK-LSK114,Fiber,dna_r10.4.1_e8.2_400bps_sup@v4.1.0-finetuned,"6mA,5mC",Dorado
2,ASA_004B,SQK-LSK114,Native,dna_r10.4.1_e8.2_400bps_sup@v4.2.0,5mCG_5hmCG,Dorado
3,ASA_005B,SQK-LSK114,Native,dna_r10.4.1_e8.2_400bps_sup@v4.1.0,5mCG_5hmCG,Dorado
4,ASA_011B,SQK-LSK114,Native,dna_r10.4.1_e8.2_400bps_sup@v5.0.0,5mC_5hmC,Dorado
...,...,...,...,...,...,...
120,ASA_186B,SQK-LSK114,Native,dna_r10.4.1_e8.2_400bps_sup@v4.2.0,5mCG_5hmCG,Dorado
121,ASA_187B,SQK-LSK114,Native,dna_r10.4.1_e8.2_400bps_sup@v4.2.0,5mCG_5hmCG,Dorado
122,ASA_188B,SQK-LSK114,Native,dna_r10.4.1_e8.2_400bps_sup@v4.2.0,5mCG_5hmCG,Dorado
123,ASA_189B,SQK-LSK114,Native,dna_r10.4.1_e8.2_400bps_sup@v4.2.0,5mCG_5hmCG,Dorado


In [7]:
merged_df = pd.merge(model, metadata, left_on='sample_name', right_on='subject_id')

In [8]:
merged_df.columns

Index(['sample_name', 'kit', 'method', 'model', 'mod_model', 'basecaller',
       'subject_id', 'source_subject_id', 'biobank_name', 'sex', 'race',
       'primary_diagnosis', 'gp2_phenotype', 'sample_id', 'subject_id_y',
       'source_sample_id', 'organism', 'tissue', 'assay_type', 'condition_id',
       'organism_ontology_term_id', 'age_at_collection'],
      dtype='object')

In [9]:
merged_df = merged_df[['sample_name','model','age_at_collection','sex', 'biobank_name']]

In [ ]:
for col in merged_df.columns:
   
    print(f"Column: {col}")
    print(merged_df[col].value_counts(dropna=False))
    print()

In [12]:
merged_df = merged_df.set_index('sample_name')
merged_df = pd.get_dummies(merged_df, columns=['sex','model','biobank_name'], drop_first=True)


In [ ]:
final = merged_df.T
final.columns = final.columns.str.replace("B", "", regex=False)
final

In [ ]:
snps_donor_pca = '../plink/WGS_chm13.snps_only_genotype_pca.eigenvec'
snps_donor_pca_df = pd.read_csv(snps_donor_pca, delim_whitespace=True)
snps_donor_pca_df.index = snps_donor_pca_df['#IID']
snps_donor_pca_df = snps_donor_pca_df.drop(columns=['#IID'])
snps_donor_pca_df = snps_donor_pca_df.T
snps_donor_pca_df.index = [f'SNP_{x}' for x in snps_donor_pca_df.index]
snps_donor_pca_df

In [ ]:
final_snp = pd.concat([final, snps_donor_pca_df], axis=0)
final_snp

In [ ]:
indel_donor_pca = '../plink/WGS_chm13.indels_only_genotype_pca.eigenvec'
indel_donor_pca_df = pd.read_csv(indel_donor_pca, delim_whitespace=True)
indel_donor_pca_df.index =indel_donor_pca_df['#IID']
indel_donor_pca_df = indel_donor_pca_df.drop(columns=['#IID'])
indel_donor_pca_df = indel_donor_pca_df.T
indel_donor_pca_df.index = [f'INDEL_{x}' for x in indel_donor_pca_df.index]

In [ ]:
final_indel = pd.concat([final, indel_donor_pca_df], axis=0)
final_indel

In [18]:
os.makedirs('./tensor_input_files', exist_ok=True)
final_snp.to_csv('./tensor_input_files/mQTL_SNP_cov.txt', sep = "\t")
final_indel.to_csv('./tensor_input_files/mQTL_indel_cov.txt', sep = "\t")

In [19]:
pip list

Package                              Version
------------------------------------ -----------------
adjustText                           1.0.4
aiohttp                              3.9.3
aiosignal                            1.3.1
anndata                              0.10.5.post1
annoy                                1.17.3
appdirs                              1.4.4
arboreto                             0.1.6
argparse-dataclass                   2.0.0
array_api_compat                     1.5.1
asttokens                            2.4.1
attr                                 0.3.2
attrs                                23.2.0
bbknn                                1.6.0
beautifulsoup4                       4.12.3
bidict                               0.23.1
bioservices                          1.11.2
blosc2                               2.5.1
bokeh                                3.4.0
boltons                              23.1.1
bs4                                  0.0.2
cattrs                     